In [ ]:
from utils.analysis.valuation import (
    CompanyAnalyzer,
    ComparisonAnalyzer,
    SectorAnalyzer,
    AnalysisWeights,
    CompanyReporter
)

In [ ]:
custom_weights = AnalysisWeights(
    profitability=0.30,
    financial_health=0.30,
    growth=0.15,
    efficiency=0.10,
    valuation=0.15
)

analyzer = CompanyAnalyzer(weights=custom_weights)
comparator = ComparisonAnalyzer(company_analyzer=analyzer)
sector_analyzer = SectorAnalyzer(company_analyzer=analyzer)
reporter = CompanyReporter()

In [ ]:
result = analyzer.analyze("META")
print(reporter.render(result))

ANÁLISIS: Meta Platforms, Inc. (META)
Sector: Communication Services | Industria: Internet Content & Information
País: United States | Moneda: USD

─────────────────────────────────────────────────────────────────
RESUMEN DE SCORES
─────────────────────────────────────────────────────────────────
  🟢 Rentabilidad       [███████████████████░]   98.8
  🟡 Salud Financiera   [███████████████░░░░░]   77.7
  🔴 Crecimiento        [███████░░░░░░░░░░░░░]   38.5
  🟠 Eficiencia         [██████████░░░░░░░░░░]   50.5
  🔴 Valoración         [████░░░░░░░░░░░░░░░░]   23.7
─────────────────────────────────────────────────────────────────
  🟡 TOTAL              [█████████████░░░░░░░]   67.3
  📋 Conclusión: BUENA

─────────────────────────────────────────────────────────────────
RENTABILIDAD (Score: 98.8)
─────────────────────────────────────────────────────────────────
  ROIC:             29.8%        excellent
  ROE:              32.6%        excellent
  ROA:              18.0%        excellent
  Marge

In [ ]:
TICKERS = ["AAPL", "MSFT", "GOOGL", "NVDA", "META"]

comparison = comparator.compare(TICKERS)
print("COMPARACIÓN DE EMPRESAS")
display(comparison['summary_df'])

COMPARACIÓN DE EMPRESAS


,Ticker,Nombre,Sector,Rentabilidad,Salud Fin.,Crecimiento,Eficiencia,Valoración,TOTAL,Conclusión
0,AAPL,Apple Inc.,Technology,100.000000,58.420135,73.250000,62.446787,11.075556,66.419553,BUENA
1,MSFT,Microsoft Corporation,Technology,96.907677,71.616439,58.687500,37.456693,15.360460,65.410098,BUENA
2,GOOGL,Alphabet Inc.,Communication S,100.000000,79.192754,70.729167,45.588734,14.403629,71.086619,BUENA
3,NVDA,NVIDIA Corporation,Technology,100.000000,90.465657,100.000000,54.294262,6.909407,78.605534,BUENA
4,META,"Meta Platforms, Inc.",Communication S,98.799440,77.683832,38.500000,50.495631,23.655010,67.317796,BUENA


In [ ]:
print("RANKING POR SCORE TOTAL")

for item in comparison['ranking']:
    emoji = "🥇" if item['rank'] == 1 else "🥈" if item['rank'] == 2 else "🥉" if item['rank'] == 3 else "  "
    print(f"{emoji} #{item['rank']} {item['ticker']:<6} {item['name'][:20]:<20} Score: {item['score']:.1f} → {item['conclusion']}")

RANKING POR SCORE TOTAL
🥇 #1 NVDA   NVIDIA Corporation   Score: 78.6 → BUENA
🥈 #2 GOOGL  Alphabet Inc.        Score: 71.1 → BUENA
🥉 #3 META   Meta Platforms, Inc. Score: 67.3 → BUENA
   #4 AAPL   Apple Inc.           Score: 66.4 → BUENA
   #5 MSFT   Microsoft Corporatio Score: 65.4 → BUENA


In [ ]:
print("MEJOR Y PEOR POR CATEGORÍA")

category_names = {
    'profitability': 'Rentabilidad',
    'financial_health': 'Salud Financiera',
    'growth': 'Crecimiento',
    'efficiency': 'Eficiencia',
    'valuation': 'Valoración'
}

for cat, data in comparison['category_leaders'].items():
    name = category_names.get(cat, cat)
    best = data['best']
    worst = data['worst']
    print(f"\n{name}:")
    print(f"  🟢 Mejor:  {best['ticker']:<6} ({best['score']:.1f})")
    print(f"  🔴 Peor:   {worst['ticker']:<6} ({worst['score']:.1f})")

MEJOR Y PEOR POR CATEGORÍA

Rentabilidad:
  🟢 Mejor:  AAPL   (100.0)
  🔴 Peor:   MSFT   (96.9)

Salud Financiera:
  🟢 Mejor:  NVDA   (90.5)
  🔴 Peor:   AAPL   (58.4)

Crecimiento:
  🟢 Mejor:  NVDA   (100.0)
  🔴 Peor:   META   (38.5)

Eficiencia:
  🟢 Mejor:  AAPL   (62.4)
  🔴 Peor:   MSFT   (37.5)

Valoración:
  🟢 Mejor:  META   (23.7)
  🔴 Peor:   NVDA   (6.9)


In [ ]:
print("POSICIÓN RELATIVA VS PEERS")

TECH_PEERS = ["AAPL", "MSFT", "GOOGL", "AMZN"]

sector_result = sector_analyzer.analyze_vs_peers(
    ticker="META",
    peers=TECH_PEERS
)

rel_pos = sector_result['relative_position']

for cat, data in rel_pos.items():
    if isinstance(data, dict) and 'company_score' in data:
        name = category_names.get(cat, cat.upper())
        diff = data['vs_avg']
        arrow = "↑" if diff > 0 else "↓" if diff < 0 else "="
        color = "🟢" if diff > 5 else "🔴" if diff < -5 else "🟡"
        
        print(f"\n{name}:")
        print(f"  Tu score:    {data['company_score']:.1f}")
        print(f"  Promedio:    {data['peer_avg']:.1f}")
        print(f"  {color} Diferencia: {diff:+.1f} {arrow}")
        print(f"  Posición:    #{data['rank']} de {data['total_compared']}")

POSICIÓN RELATIVA VS PEERS

Rentabilidad:
  Tu score:    98.8
  Promedio:    90.0
  🟢 Diferencia: +8.8 ↑
  Posición:    #3 de 5

Salud Financiera:
  Tu score:    77.7
  Promedio:    68.5
  🟢 Diferencia: +9.2 ↑
  Posición:    #4 de 5

Crecimiento:
  Tu score:    38.5
  Promedio:    68.0
  🔴 Diferencia: -29.5 ↓
  Posición:    #1 de 5

Eficiencia:
  Tu score:    50.5
  Promedio:    54.0
  🟡 Diferencia: -3.5 ↓
  Posición:    #3 de 5

Valoración:
  Tu score:    23.7
  Promedio:    16.4
  🟢 Diferencia: +7.3 ↑
  Posición:    #4 de 5

TOTAL:
  Tu score:    67.3
  Promedio:    65.6
  🟡 Diferencia: +1.7 ↑
  Posición:    #4 de 5
